In [ ]:
# Cell 1: Optimized Structure
import veloxchem as vlx
import os
os.environ["OMP_NUM_THREADS"] = "8"  # <-- set once

pdb_path = 'step_1_1_RS.pdb'  # CHANGE to your PDB
molecule = vlx.Molecule.read_pdb_file(pdb_path)

basis = vlx.MolecularBasis.read(molecule, "def2-svp")
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"          # CHANGE functional if needed (e.g. "PBE0", "B3LYP-D3BJ")
scf_drv.conv_thresh = 1e-8
scf_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.ostream.mute()
opt_drv.transition = False       # False = minimum (use True for TS)
opt_drv.hessian = "first+last"   # best for stability on 60-atom systems

opt_results = opt_drv.compute(molecule, basis)
final_mol = vlx.Molecule.read_xyz_string(opt_results["final_geometry"])

print(f"✅ Optimization converged! Final energy: {scf_drv.get_scf_energy():.6f} Hartree")
print("Optimized geometry written to optimized_state1.xyz")
final_mol.write_xyz_file("optimized_state1.xyz")

In [ ]:
# Cell 2: QM Barrier (example — change file paths for each state)
import veloxchem as vlx

# Optimized reactant
molecule = vlx.Molecule.read_xyz_file("optimized_state1.xyz")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.conv_thresh = 1e-8
scf_drv.ostream.mute()

scf_results = scf_drv.compute(molecule, basis)
reactant_energy = scf_drv.get_scf_energy()

# Repeat for every other state (product, intermediates, TS)
# Example for product:
# molecule_prod = vlx.Molecule.read_xyz_file("optimized_stateX.xyz")
# ... same settings ...
# product_energy = scf_drv.get_scf_energy()

# Barrier (in Hartree; convert to kcal/mol if you want)
# barrier = abs(reactant_energy - product_energy) * 627.509
print(f"Reactant QM energy: {reactant_energy:.6f} Hartree")
# print(f"Barrier: {barrier:.2f} kcal/mol")

In [ ]:
# Cell 3: Normal Modes (use any optimized geometry)
import veloxchem as vlx
import numpy as np

molecule = vlx.Molecule.read_xyz_file("optimized_state1.xyz")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.ostream.mute()

scf_results = scf_drv.compute(molecule, basis)
energy = scf_drv.get_scf_energy()

vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_results = vib_drv.compute(molecule, basis)

print("Vibrational frequencies (cm⁻¹):")
for freq in vib_results['frequencies']:
    print(f"  {freq:.2f}")

normal_modes = vib_results['normal_modes']
print(f"\nNumber of modes: {normal_modes.shape[1]}")
# Print first mode (bohr/amu^{1/2})
print("First normal mode (example):")
print(normal_modes[:, 0])

In [ ]:
# Cell 4: QM Charges - RESP + ESP (for any optimized state - e.g. optimized_state1.xyz)
import veloxchem as vlx
import os
os.environ["OMP_NUM_THREADS"] = "8"  # keep your CPU threads

# Optimized geometry (change path for each state)
molecule = vlx.Molecule.read_xyz_file("optimized_state1.xyz")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")  # or tighter if needed
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.conv_thresh = 1e-8
scf_drv.ostream.mute()
scf_results = scf_drv.compute(molecule, basis)

print(f"Total charge from SCF: {scf_results['total_charge']:.4f}")

# 1. RESP charges (recommended for force fields / MM)
resp_drv = vlx.RespChargesDriver()
resp_drv.equal_charges = ""          # leave empty = no constraints (or "1=3, 1=4" for equivalent atoms)
# Optional tuning (recommended defaults for ~60 atoms):
# resp_drv.n_layers = 4
# resp_drv.points_per_ang2 = 1.0
resp_charges = resp_drv.compute(molecule, basis, scf_results)  # no "esp" arg needed

print("\n=== RESP Charges (e) ===")
for label, charge in zip(molecule.get_labels(), resp_charges):
    print(f"  {label:2s}: {charge:10.6f}")
print(f"Total RESP charge: {resp_charges.sum():.4f}")

# 2. ESP charges (Merz-Kollman / CHELPG)
esp_drv = vlx.EspChargesDriver()
esp_drv.grid_type = "chelpg"          # or "mk" for pure MK
esp_drv.equal_charges = ""            # no constraints
esp_charges = esp_drv.compute(molecule, basis, scf_results)

print("\n=== ESP / CHELPG Charges (e) ===")
for label, charge in zip(molecule.get_labels(), esp_charges):
    print(f"  {label:2s}: {charge:10.6f}")
print(f"Total ESP charge: {esp_charges.sum():.4f}")